In [13]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
import re

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 120


# Pattern to match all relevant CSV files
file_pattern = path + "negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low*.csv"

# List all matching files
csv_files = glob.glob(file_pattern)

for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)

/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.8.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low2.5.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low0.5.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low3.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low1.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_dim4_ncycles1200_gamma1e-06_H_high3.0_H_low1.5.csv
[2.8, 2.0, 2.5, 0.5, 3.0, 1.0, 1.5]


In [36]:
#Correlations
path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
# Pattern to match all relevant CSV files
file_pattern = path + "decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low*.csv"

# List all matching files
csv_files = glob.glob(file_pattern)
# /Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low0.5.csv
for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)


/Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low0.5.csv
/Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low2.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low1.0.csv
[0.5, 2.0, 1.0]


In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

H_low = 0.5
path = '/Users/jiakai/Desktop/SURF/code/_spirit/'

path_chi = path + f"negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low{H_low}.csv"
path_corr = path + f"decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles1200_gamma1e-06_H_high3.0_H_low{H_low}.csv"


# Read CSVs
df_chi = pd.read_csv(path_chi)
df_corr = pd.read_csv(path_corr)

# Group and compute mean & SEM
df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()

df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

# Group correlation by k and calculate mean + SEM
df_corr_avg = (
    df_corr.groupby("k", as_index=False)
    .agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )
)

# ---- Create subplot with secondary y-axis ----
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Susceptibility trace (left y-axis)
fig.add_trace(
    go.Scatter(
        x=df_chi_avg["i"],
        y=df_chi_avg["chi_mean"],
        error_y=dict(
            type="data",
            array=df_chi_avg["chi_sem"],
            visible=True
        ),
        mode='lines+markers',
        name=f"Chi H_low = {H_low}" if H_low != 3.0 else "Ref (H_low = H_high)",
        hovertemplate=f"H_low={H_low}<br>i=%{{x}}<br>χ=%{{y:.3f}}<extra></extra>"
    ),
)

# Correlation trace (right y-axis)
fig.add_trace(
    go.Scatter(
        x=df_corr_avg["k"],
        y=df_corr_avg["corr_avg"],
        error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
        mode='lines+markers',
        name=f"Spin Correlation, H_low{H_low}",
        # line=dict(color='red')
    ),
    secondary_y=True
)

# Axis titles
fig.update_xaxes(title_text="MCS Step")
fig.update_yaxes(title_text="Susceptibility χ", secondary_y=False)
fig.update_yaxes(title_text="Average Correlation", secondary_y=True)

# Layout
fig.update_layout(
    title="Susceptibility and Correlation vs Field / k",
    legend=dict(x=0.02, y=0.98),
    template="plotly_white",
)

# Save figure
fig.write_html(f"combined_dual_axis_plot_{H_low}.html")



In [38]:
fig.show()

In [39]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 1200
H_low_values = [0.5, 1.0, 2.0]  # Add more if you want

# ---- Create subplot with secondary y-axis ----
fig = make_subplots(specs=[[{"secondary_y": True}]])

for H_low in H_low_values:
    # File paths
    path_chi = path + f"negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"
    path_corr = path + f"decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"

    # Read CSVs
    df_chi = pd.read_csv(path_chi)
    df_corr = pd.read_csv(path_corr)

    # Group chi data
    df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()
    df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

    # Group correlation data
    df_corr_avg = df_corr.groupby("k", as_index=False).agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )

    # Susceptibility trace (left y-axis)
    fig.add_trace(
        go.Scatter(
            x=df_chi_avg["i"],
            y=df_chi_avg["chi_mean"],
            error_y=dict(type="data", array=df_chi_avg["chi_sem"], visible=True),
            mode='lines+markers',
            name=f"Chi H_low={H_low}",
            hovertemplate=f"H_low={H_low}<br>i=%{{x}}<br>χ=%{{y:.3f}}<extra></extra>"
        ),
        secondary_y=False
    )

    # Correlation trace (right y-axis)
    fig.add_trace(
        go.Scatter(
            x=df_corr_avg["k"],
            y=df_corr_avg["corr_avg"],
            error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
            mode='lines+markers',
            name=f"Corr H_low={H_low}"
        ),
        secondary_y=True
    )

# Axis titles
fig.update_xaxes(title_text="MCS Step")
fig.update_yaxes(title_text="Susceptibility χ", secondary_y=False)
fig.update_yaxes(title_text="Average Correlation", secondary_y=True)

# Layout
fig.update_layout(
    title="Susceptibility and Correlation vs Field/k for Multiple H_low",
    legend=dict(x=0.02, y=0.98),
    template="plotly_white",
)

# Save figure
fig.write_html("combined_dual_axis_plot_multiple_H_low.html")


In [40]:
fig.show()

In [51]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 1200
H_low_values = [0.5, 1.0, 2.0, 2.5]  # Add more if needed

# Create stacked subplots with shared x-axis alignment
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Susceptibility χ", "Average Correlation")
)

for H_low in H_low_values:
    # File paths
    path_chi = path + f"negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"
    path_corr = path + f"decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"

    # Read CSVs
    df_chi = pd.read_csv(path_chi)
    df_corr = pd.read_csv(path_corr)

    # Group chi data
    df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()
    df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

    # Group correlation data
    df_corr_avg = df_corr.groupby("k", as_index=False).agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )

    # Susceptibility (top)
    fig.add_trace(
        go.Scatter(
            x=df_chi_avg["i"],
            y=df_chi_avg["chi_mean"],
            error_y=dict(type="data", array=df_chi_avg["chi_sem"], visible=True),
            mode='lines+markers',
            name=f"Chi H_low={H_low}"
        ),
        row=1, col=1
    )

    # Correlation (bottom)
    fig.add_trace(
        go.Scatter(
            x=df_corr_avg["k"],
            y=df_corr_avg["corr_avg"],
            error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
            mode='lines+markers',
            name=f"Correlation H_low={H_low}"
        ),
        row=2, col=1
    )

# Axis labels
fig.update_xaxes(title_text="MCS Step", row=2, col=1)
fig.update_yaxes(title_text="Susceptibility χ", row=1, col=1)
fig.update_yaxes(title_text="Average Correlation", row=2, col=1)

# Layout
fig.update_layout(
    # height=800,
    title="Susceptibility and Correlation vs MCS",
    template="plotly_white",
    legend=dict(x=0.02, y=0.02)
)

# Save
fig.write_html("stacked_susceptibility_correlation.html")


In [52]:
fig.show()

In [58]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 1200
H_low_values = [0.5, 1.0, 2.0, 2.5]  # Add more if needed

# Pick a distinct color for each H_low
color_map = {H: px.colors.qualitative.Plotly[i] for i, H in enumerate(H_low_values)}

# Create stacked subplots with shared x-axis alignment
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Susceptibility χ", "Average Correlation")
)

for H_low in H_low_values:
    # File paths
    path_chi = path + f"negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"
    path_corr = path + f"decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"

    # Read CSVs
    df_chi = pd.read_csv(path_chi)
    df_corr = pd.read_csv(path_corr)

    # Group chi data
    df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()
    df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

    # Group correlation data
    df_corr_avg = df_corr.groupby("k", as_index=False).agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )

    # Susceptibility (top) — same color as correlation for same H_low
    fig.add_trace(
        go.Scatter(
            x=df_chi_avg["i"],
            y=df_chi_avg["chi_mean"],
            error_y=dict(type="data", array=df_chi_avg["chi_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Chi H_low={H_low}"
        ),
        row=1, col=1
    )

    # Correlation (bottom) — same color as chi
    fig.add_trace(
        go.Scatter(
            x=df_corr_avg["k"],
            y=df_corr_avg["corr_avg"],
            error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Correlation H_low={H_low}"
        ),
        row=2, col=1
    )

# Axis labels
fig.update_xaxes(title_text="MCS Step", row=2, col=1)
fig.update_yaxes(title_text="Susceptibility", row=1, col=1)
fig.update_yaxes(title_text="Average Correlation", row=2, col=1)

# Layout
fig.update_layout(
    height=800,
    title="Susceptibility and Correlation vs Field",
    template="plotly_white",
    legend=dict(x=0.02, y=0.02)
)

# Save
fig.write_html("stacked_susceptibility_correlation.html")


In [59]:
fig.show()

In [61]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 1200
H_low_values = [1.0, 2.0]  # Add more if needed

# Pick a distinct color for each H_low
color_map = {H: px.colors.qualitative.Plotly[i] for i, H in enumerate(H_low_values)}

# Create stacked subplots with shared x-axis alignment
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("Susceptibility χ", "Average Correlation")
)

for H_low in H_low_values:
    # File paths
    path_chi = path + f"40_negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"
    path_corr = path + f"40_decay_correlations_negative_field_cycle_dim4_anisotropy0.7_ncycles{n_cycles}_gamma1e-06_H_high3.0_H_low{H_low}.csv"

    # Read CSVs
    df_chi = pd.read_csv(path_chi)
    df_corr = pd.read_csv(path_corr)

    # Group chi data
    df_chi_avg = df_chi.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()
    df_chi_avg['chi_sem'] = df_chi_avg['chi_std'] / (n_cycles ** 0.5)

    # Group correlation data
    df_corr_avg = df_corr.groupby("k", as_index=False).agg(
        corr_avg=("corr", "mean"),
        corr_sem=("corr", lambda x: x.std(ddof=1) / (len(x) ** 0.5))
    )

    # Susceptibility (top) — same color as correlation for same H_low
    fig.add_trace(
        go.Scatter(
            x=df_chi_avg["i"],
            y=df_chi_avg["chi_mean"],
            error_y=dict(type="data", array=df_chi_avg["chi_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Chi H_low={H_low}"
        ),
        row=1, col=1
    )

    # Correlation (bottom) — same color as chi
    fig.add_trace(
        go.Scatter(
            x=df_corr_avg["k"],
            y=df_corr_avg["corr_avg"],
            error_y=dict(type='data', array=df_corr_avg["corr_sem"], visible=True),
            mode='lines+markers',
            marker=dict(color=color_map[H_low]),
            line=dict(color=color_map[H_low]),
            name=f"Correlation H_low={H_low}"
        ),
        row=2, col=1
    )

# Axis labels
fig.update_xaxes(title_text="MCS Step", row=2, col=1)
fig.update_yaxes(title_text="Susceptibility", row=1, col=1)
fig.update_yaxes(title_text="Average Correlation", row=2, col=1)

# Layout
fig.update_layout(
    height=800,
    title="Susceptibility and Correlation vs Field",
    template="plotly_white",
    legend=dict(x=0.02, y=0.02)
)

# Save
fig.write_html("40_stacked_susceptibility_correlation.html")

In [62]:
fig.show()